In [1]:
from src.database.connector import db_connector
from src.database.structure import ShopeeOrder, ComboVariant, Combo, Variant
from pathlib import Path
from sqlalchemy import select

In [2]:
BASE_PATH = Path().cwd().parent
DB_PATH = Path(BASE_PATH / "database.sqlite3")

session = db_connector(DB_PATH)

In [3]:
combo_variant_pairs = select(ShopeeOrder.combo_name, ShopeeOrder.variant_name).distinct()

unq_set = {
    (cv.combo_name, cv.variant_name) for cv in
    session.execute(combo_variant_pairs).all()
}
unq_list = list(unq_set)


unq_list = list(unq_set)
input_combos = {c[0] for c in unq_list if c[0]}
input_variant = {v[1] for v in unq_list if v[1] and v[1].strip()}

In [4]:
exist_combos = {
    c.combo_name: c
    for c in session.scalars(select(Combo).where(Combo.combo_name.in_(input_combos))).all()
}

exist_variant = {
    v.variant_name: v
    for v in session.scalars(select(Variant).where(Variant.variant_name.in_(input_variant))).all()
}

created_cv_pair = {
    (cv.combo_key, cv.variant_key)
    for cv in session.scalars(select(ComboVariant)).all()
}

created_cv_pair = {
    (cv.combo_key, cv.variant_key)
    for cv in session.scalars(select(ComboVariant)).all()
}

for combo_name, variant_name in unq_list:
    if combo_name in exist_combos:
        combo_obj = exist_combos[combo_name]
    else:
        combo_obj = Combo(combo_name=combo_name)
        session.add(combo_obj)
        print(f"Thêm mới Combo: {combo_obj.combo_name}")
        exist_combos[combo_name] = combo_obj

    variant_obj = None

    if variant_name:
        if variant_name in exist_variant:
            variant_obj = exist_variant[variant_name]
        else:
            variant_obj = Variant(variant_name=variant_name)
            session.add(variant_obj)
            print(f"Thêm mới Variant: {variant_obj.variant_name}")
            exist_variant[variant_name] = variant_obj

    session.flush()

    current_cv_pair = (combo_obj.combo_key, variant_obj.variant_key if variant_obj else None)

    if current_cv_pair not in created_cv_pair:
        combo_variant_obj = ComboVariant(combo_link=combo_obj, variant_link=variant_obj)
        session.add(combo_variant_obj)
        created_cv_pair.add(current_cv_pair)
        print(f"Đã tạo cặp {combo_obj.combo_name, variant_name}")
    else:
        pass

try:
    session.commit()
    print("Import dữ liệu thành công mỹ mãn!")
except Exception as e:
    session.rollback()
    print(e)

Import dữ liệu thành công mỹ mãn!
